## Theory on Handling Missing Values

Missing values are a common problem in real-world datasets and can significantly impact the quality of analysis and machine learning models. Handling them effectively is a crucial step in data preprocessing. Here's a breakdown of common strategies:

### Why do we have missing values?

Missing data can arise for various reasons:
*   **Data entry errors**: Mistakes during manual data input.
*   **Data collection issues**: Equipment malfunction, survey non-response, or unrecorded observations.
*   **Data integration problems**: Mismatched or non-existent records when combining data from different sources.
*   **Intentional omission**: Some values might be irrelevant or not applicable.

### Common Strategies for Handling Missing Values

1.  **Deletion Methods**:
    *   **Listwise Deletion (or Complete Case Analysis)**: Rows containing any missing values are entirely removed. This is simple but can lead to significant loss of data if many rows have missing values, potentially introducing bias if the missingness is not random.
    *   **Pairwise Deletion**: Only removes rows with missing values for the specific variables being analyzed in a particular calculation. This uses more data than listwise deletion but can make statistical comparisons inconsistent as different analyses might be based on different subsets of data.

2.  **Imputation Methods**:
    Imputation involves filling in missing values with estimated or substituted values. This preserves more data but introduces some level of artificiality.

    *   **Mean/Median/Mode Imputation**: Replacing missing numerical values with the mean or median of the existing values in that column. For categorical values, the mode (most frequent category) is often used. This is simple and fast, but it reduces variance and doesn't account for relationships between variables.

    *   **Forward Fill (ffill) / Backward Fill (bfill)**: Filling missing values with the previous or next valid observation in a sequence, often used for time-series data.

    *   **Regression Imputation**: As demonstrated in the current notebook, this method uses a regression model to predict missing values based on other variables in the dataset. For each variable with missing values, a regression equation is built using available data, and then this equation is used to predict the missing entries. This is more sophisticated than mean/median imputation as it considers relationships between variables, but it assumes a linear relationship and can underestimate the variance.
        *   **How it works (as seen in `df_missing`)**: We train a `LinearRegression` model using the `Cr` (Chromium) values as features (`X_train`) and the known `M` (Manganese) values as the target (`y_train`). Then, for the missing `M` value, we use its corresponding `Cr` value (`X_predict`) to predict what `M` should be. This predicted value is then used to fill the `None` entry.

    *   **K-Nearest Neighbors (KNN) Imputation**: Replaces missing values with the average (for numerical data) or mode (for categorical data) of the K-nearest neighbors. It works well with non-linear relationships and different data types but can be computationally expensive for large datasets.

    *   **Advanced Imputation (e.g., MICE - Multiple Imputation by Chained Equations)**: This method imputes missing values multiple times, creating several complete datasets. Each dataset is then analyzed, and the results are combined to produce a single, more robust estimate. MICE is complex but provides more accurate estimates of uncertainty.

### Choosing an Imputation Method

The best method depends on several factors:
*   **Nature of Missingness**: Is the data missing completely at random (MCAR), missing at random (MAR), or missing not at random (MNAR)?
*   **Type of Data**: Numerical, categorical, or time-series.
*   **Proportion of Missing Data**: Small amounts of missing data might allow simpler methods, while large amounts require more sophisticated approaches.
*   **Impact on Analysis**: Consider how each method might affect downstream analysis or model performance.

In our example, using linear regression imputation allowed us to leverage the observed relationship between 'Cr' and 'M' to make an informed estimate for the missing 'M' value, preserving the data point rather than discarding the entire row.

In [1]:
import numpy as np
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import pandas as pd

data = {
    'Cr': [2, 4, 7, 9, 11],
    'M': [0, 1, 2, 3, None]
}

# Create a DataFrame from the data
df_missing = pd.DataFrame(data)
print("Original DataFrame with missing value:")
display(df_missing)

# Identify the missing value
missing_index = df_missing[df_missing['M'].isnull()].index[0]
X_predict = df_missing.loc[missing_index, 'Cr'].reshape(-1, 1)

# Separate data into known and unknown for M
df_known = df_missing.dropna()
X_train = df_known['Cr'].values.reshape(-1, 1)
y_train = df_known['M'].values

# Initialize and train the Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Predict the missing M value
predicted_M = model.predict(X_predict)[0]

# Fill the missing value in the DataFrame
df_missing.loc[missing_index, 'M'] = predicted_M

print("\nDataFrame after filling missing value with linear regression prediction:")
display(df_missing)

Original DataFrame with missing value:


,Cr,M
0,2,0.0
1,4,1.0
2,7,2.0
3,9,3.0
4,11,NaN



DataFrame after filling missing value with linear regression prediction:


,Cr,M
0,2,0.000000
1,4,1.000000
2,7,2.000000
3,9,3.000000
4,11,3.775862


In [2]:
from sklearn.linear_model import LinearRegression

# Use the df_missing DataFrame with the filled value
X = df_missing['Cr'].values.reshape(-1, 1)
y = df_missing['M'].values

model = LinearRegression()
model.fit(X, y)

print(f"Coefficient: {model.coef_[0]:.2f}")
print(f"Intercept: {model.intercept_:.2f}")

Coefficient: 0.41
Intercept: -0.78


Build a linear Regration model.</br>
Predict the final score after.</br>
13 over.</br>
20 over.</br>

In [3]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

data={
    'Overs':[2,6,8,12,17],
    'Runs':[18,52,74,112,153]
}

In [4]:
df_new_data = pd.DataFrame(data)

# Define features (X) and target (y) for the new model
X_train_new = df_new_data['Overs'].values.reshape(-1, 1)
y_train_new = df_new_data['Runs'].values

# Initialize and train a new Linear Regression model
new_model = LinearRegression()
new_model.fit(X_train_new, y_train_new)

X_predict_new = np.array([13, 20]).reshape(-1, 1)
predicted_scores = new_model.predict(X_predict_new)

print(f"Predicted score for 13 overs: {predicted_scores[0]:.2f}")
print(f"Predicted score for 20 overs: {predicted_scores[1]:.2f}")

Predicted score for 13 overs: 118.28
Predicted score for 20 overs: 182.13
